# Silver Exploration Notebook — Open Brewery DB

This notebook profiles the **Silver layer** to validate the current transformations.

In [ ]:
%pip install pandas

from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)


## 1) Locate a Silver snapshot

In [ ]:
candidates = sorted(Path("../data/silver/breweries").glob("ingestion_date=*/run_id=*"))
SILVER_ROOT = candidates[-1] if candidates else None

SILVER_ROOT


In [ ]:
if SILVER_ROOT is None or not SILVER_ROOT.exists():
    raise FileNotFoundError("No Silver snapshot found. Set SILVER_ROOT manually or run the Silver job first.")

PARQUET_FILES = sorted(SILVER_ROOT.rglob("*.parquet"))
if not PARQUET_FILES:
    raise FileNotFoundError(f"No parquet files found under {SILVER_ROOT}")

len(PARQUET_FILES), PARQUET_FILES[:10]


## 2) Load Silver

In [ ]:
df = pd.concat([pd.read_parquet(f) for f in PARQUET_FILES], ignore_index=True)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
df.head(5)


## 3) Schema, nulls, cardinality

In [ ]:
schema_df = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "non_null": [int(df[c].notna().sum()) for c in df.columns],
    "nulls": [int(df[c].isna().sum()) for c in df.columns],
    "null_pct": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
    "n_unique": [int(df[c].nunique(dropna=True)) for c in df.columns],
}).sort_values(["null_pct", "column"], ascending=[False, True])

schema_df


## 4) Key dimensions and `unknown` usage

In [ ]:
key_dims = [c for c in ["country", "state_province", "city", "brewery_type"] if c in df.columns]

def profile_dimension(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip()
    return pd.Series({
        "nulls": int(s.isna().sum()),
        "unknown_count": int((s.str.lower() == "unknown").fillna(False).sum()),
        "n_unique": int(s.nunique(dropna=True)),
    })

dim_profile = pd.DataFrame({c: profile_dimension(df[c]) for c in key_dims}).T
dim_profile["unknown_pct"] = (dim_profile["unknown_count"] / len(df) * 100).round(2)
dim_profile


In [ ]:
for col in key_dims:
    print(f"\n=== Top values: {col} ===")
    display(df[col].astype("string").value_counts(dropna=False).head(20).to_frame("count"))


## 5) Numeric casting quality

In [ ]:
for col in [c for c in ["latitude", "longitude"] if c in df.columns]:
    parsed = pd.to_numeric(df[col], errors="coerce")
    print(f"\n=== {col} ===")
    print({
        "non_null": int(df[col].notna().sum()),
        "numeric_non_null": int(parsed.notna().sum()),
        "failed_casts": int(df[col].notna().sum() - parsed.notna().sum()),
        "min": parsed.min(),
        "max": parsed.max(),
    })


## 6) Duplicate and business-key checks

In [ ]:
if "id" in df.columns:
    print("Duplicated ids:", int(df["id"].duplicated().sum()))
else:
    print("Column `id` not found in Silver.")


In [ ]:
candidate_keys = [c for c in ["country", "state_province", "city", "brewery_type", "name"] if c in df.columns]
if candidate_keys:
    dup_candidate = df.duplicated(subset=candidate_keys).sum()
    print("Potential duplicated business rows using candidate dimensions:", int(dup_candidate))
else:
    print("No candidate business-key dimensions available.")


## 7) Partition sanity check

In [ ]:
partition_stats = (
    df.groupby(["country", "state_province"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)

print("Total partitions:", len(partition_stats))
display(partition_stats.head(20))


In [ ]:
# Compare folder names with analytical values to spot sanitization mismatches
folder_partitions = []
for path in SILVER_ROOT.glob("country=*/state_province=*"):
    folder_partitions.append({
        "country_partition": path.parent.name.split("=", 1)[1],
        "state_partition": path.name.split("=", 1)[1],
    })

folder_partitions_df = pd.DataFrame(folder_partitions).drop_duplicates().sort_values(
    ["country_partition", "state_partition"]
)
folder_partitions_df.head(20)


## 8) Encoding / replacement-character inspection

In [ ]:
def contains_replacement_char(value) -> bool:
    if pd.isna(value):
        return False
    return "\ufffd" in str(value)

text_cols = [c for c in df.columns if str(df[c].dtype) in ("object", "string")]
replacement_summary = pd.DataFrame([
    {
        "column": c,
        "replacement_char_rows": int(df[c].map(contains_replacement_char).sum()),
    }
    for c in text_cols
]).sort_values(["replacement_char_rows", "column"], ascending=[False, True])

replacement_summary


In [ ]:
suspects = []
for c in text_cols:
    mask = df[c].map(contains_replacement_char)
    if mask.any():
        suspects.append({
            "column": c,
            "sample_values": df.loc[mask, c].astype("string").dropna().head(10).tolist()
        })

pd.DataFrame(suspects) if suspects else "No replacement-character issues detected in Silver."
